# Análise de Criminalidade em Grandes Cidades Brasileiras
**Projeto G2 — Tema 15 | Linguagem de Programação**

Análise exploratória de dados de criminalidade no Brasil entre 2015 e 2024,
utilizando Python, Pandas, Matplotlib, Seaborn e SQLAlchemy.

## 1. Introdução

A criminalidade urbana é um dos principais desafios da segurança pública brasileira.
Este notebook realiza uma análise exploratória completa sobre ocorrências, vítimas,
prisões e índices de violência em grandes cidades do Brasil, com o objetivo de
identificar padrões, tendências e fatores associados à criminalidade.

## 2. Contextualização

O Brasil apresenta altos índices de criminalidade concentrados nos grandes centros urbanos.
Fatores como desigualdade de renda, urbanização acelerada e déficits em políticas públicas
contribuem para a complexidade do cenário. A análise de dados criminais permite embasar
decisões estratégicas de segurança pública com evidências concretas.

## 3. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlalchemy as sa
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "#0b0e17"
plt.rcParams["axes.facecolor"] = "#111520"
plt.rcParams["text.color"] = "#e2e8f0"
plt.rcParams["axes.labelcolor"] = "#8892a4"
plt.rcParams["xtick.color"] = "#8892a4"
plt.rcParams["ytick.color"] = "#8892a4"
plt.rcParams["axes.edgecolor"] = "#1e2535"
plt.rcParams["grid.color"] = "#1e2535"

ROXO = "#7c3aed"
ROXO_LT = "#a78bfa"
VERMELHO = "#ef4444"
VERDE = "#4ade80"
CORES = ["#7c3aed","#a78bfa","#6d28d9","#ef4444","#fbbf24","#4ade80"]

print("✅ Bibliotecas importadas com sucesso")

## 4. Leitura dos Dados

In [ ]:
df = pd.read_csv("dados/simulacao_criminalidade_brasil.csv", encoding="utf-8-sig")
df["data"] = pd.to_datetime(df["data"])
print(f"Shape: {df.shape}")
df.head()

## 5. Limpeza e Preparação dos Dados

In [ ]:
# Verificar valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())
print()
print("Tipos de dados:")
print(df.dtypes)

In [ ]:
# Verificar duplicatas
duplicatas = df.duplicated().sum()
print(f"Linhas duplicadas: {duplicatas}")

# Garantir tipos corretos
df["ano"] = df["ano"].astype(int)
df["mes"] = df["mes"].astype(int)
df["ocorrencias"] = df["ocorrencias"].astype(int)
df["vitimas"] = df["vitimas"].astype(int)
df["prisoes"] = df["prisoes"].astype(int)

print("
✅ Dados prontos para análise")

## 6. Engenharia de Atributos

In [ ]:
# Taxa de prisão por ocorrência
df["taxa_prisao"] = (df["prisoes"] / df["ocorrencias"].replace(0, np.nan)).round(3)

# Faixa de renda categorizada
df["faixa_renda"] = pd.cut(
    df["renda_media"],
    bins=[0, 2000, 3500, 5000, 99999],
    labels=["Baixa", "Média", "Média-Alta", "Alta"]
)

# Semestre
df["semestre"] = df["mes"].apply(lambda m: "1º Sem" if m <= 6 else "2º Sem")

print("✅ Novas colunas criadas:")
print(["taxa_prisao", "faixa_renda", "semestre"])
df[["taxa_prisao","faixa_renda","semestre"]].head()

## 7. KPIs Principais

In [ ]:
total_oc     = df["ocorrencias"].sum()
total_vit    = df["vitimas"].sum()
total_pris   = df["prisoes"].sum()
idx_medio    = round(df["indice_violencia"].mean(), 1)
cidade_crit  = df.groupby("cidade")["ocorrencias"].sum().idxmax()
crime_freq   = df.groupby("tipo_crime")["ocorrencias"].sum().idxmax()
regiao_crit  = df.groupby("regiao")["ocorrencias"].sum().idxmax()
cidades_n    = df["cidade"].nunique()

kpis = {
    "Total de Ocorrências":       f"{total_oc:,}",
    "Total de Vítimas":           f"{total_vit:,}",
    "Total de Prisões":           f"{total_pris:,}",
    "Índice Médio de Violência":  str(idx_medio),
    "Cidade Mais Crítica":        cidade_crit,
    "Crime Mais Frequente":       crime_freq,
    "Região Mais Crítica":        regiao_crit,
    "Cidades Monitoradas":        str(cidades_n),
}

print("=" * 40)
print("  KPIs — CRIMINALIDADE NO BRASIL")
print("=" * 40)
for k, v in kpis.items():
    print(f"  {k:<30} {v}")
print("=" * 40)

## 8. Visualizações

### 8.1 Evolução Anual de Ocorrências

In [ ]:
ev = df.groupby("ano")["ocorrencias"].sum().reset_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ev["ano"], ev["ocorrencias"], color=ROXO, linewidth=2.5, marker="o",
        markersize=7, markerfacecolor=ROXO_LT)
ax.fill_between(ev["ano"], ev["ocorrencias"], alpha=0.12, color=ROXO)
ax.set_title("Evolução Anual de Ocorrências", color="#e2e8f0", fontsize=14, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.set_xticks(ev["ano"])
plt.tight_layout()
plt.show()

### 8.2 Top 10 Cidades Mais Críticas

In [ ]:
rank = (df.groupby("cidade")["ocorrencias"].sum()
        .reset_index().sort_values("ocorrencias", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(rank["cidade"], rank["ocorrencias"], color=ROXO)
for bar, val in zip(bars, rank["ocorrencias"]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", color="#fbbf24", fontsize=11)
ax.invert_yaxis()
ax.set_title("Top 10 Cidades — Total de Ocorrências", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 8.3 Ocorrências por Tipo de Crime

In [ ]:
tc = (df.groupby("tipo_crime")["ocorrencias"].sum()
      .reset_index().sort_values("ocorrencias", ascending=False))

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(tc["tipo_crime"], tc["ocorrencias"],
              color=CORES[:len(tc)])
for bar, val in zip(bars, tc["ocorrencias"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f"{val:,}", ha="center", color="#fbbf24", fontsize=11)
ax.set_title("Ocorrências por Tipo de Crime", color="#e2e8f0", fontsize=14, pad=12)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### 8.4 Heatmap — Crime × Período do Dia

In [ ]:
hc = df.groupby(["tipo_crime","periodo_dia"])["ocorrencias"].sum().reset_index()
hc_piv = hc.pivot(index="tipo_crime", columns="periodo_dia", values="ocorrencias").fillna(0)
ordem = [p for p in ["Madrugada","Manhã","Tarde","Noite"] if p in hc_piv.columns]
hc_piv = hc_piv[ordem]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(hc_piv, annot=True, fmt=".0f", cmap="RdPu",
            linewidths=0.5, linecolor="#0b0e17",
            ax=ax, cbar=False,
            annot_kws={"size":11, "color":"#e2e8f0"})
ax.set_title("Heatmap — Tipo de Crime × Período do Dia", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 8.5 Dispersão — Renda Média × Índice de Violência

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    df["renda_media"], df["indice_violencia"],
    c=df["ocorrencias"], cmap="RdPu",
    alpha=0.4, s=15, edgecolors="none"
)
# Linha de tendência
m, b = np.polyfit(df["renda_media"], df["indice_violencia"], 1)
xs = np.linspace(df["renda_media"].min(), df["renda_media"].max(), 200)
ax.plot(xs, m*xs + b, color=ROXO_LT, linewidth=1.8, linestyle="--", label="Tendência")
ax.set_title("Renda Média × Índice de Violência", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("Renda Média (R$)")
ax.set_ylabel("Índice de Violência")
ax.legend()
plt.tight_layout()
plt.show()

corr = df[["renda_media","indice_violencia"]].corr().iloc[0,1]
print(f"Correlação de Pearson: {corr:.4f}")

### 8.6 Ocorrências por Região

In [ ]:
reg = (df.groupby("regiao")["ocorrencias"].sum()
       .reset_index().sort_values("ocorrencias", ascending=False))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(reg["regiao"], reg["ocorrencias"],
              color=[ROXO, ROXO_LT, "#6d28d9", VERMELHO, "#fbbf24"][:len(reg)])
for bar, val in zip(bars, reg["ocorrencias"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{val:,}", ha="center", color="#fbbf24", fontsize=11)
ax.set_title("Ocorrências por Região", color="#e2e8f0", fontsize=14, pad=12)
plt.tight_layout()
plt.show()

## 9. Persistência com SQLAlchemy e SQLite

In [ ]:
engine = sa.create_engine("sqlite:///database/criminalidade.db", echo=False)

# Persiste o DataFrame principal
df.to_sql("ocorrencias", engine, if_exists="replace", index=False)
print("✅ Tabela 'ocorrencias' gravada com sucesso")

# Tabela de KPIs
kpis_df = pd.DataFrame(list(kpis.items()), columns=["indicador","valor"])
kpis_df.to_sql("kpis", engine, if_exists="replace", index=False)
print("✅ Tabela 'kpis' gravada com sucesso")

In [ ]:
# Consulta de exemplo com SQL
with engine.connect() as conn:
    resultado = pd.read_sql(
        "SELECT cidade, SUM(ocorrencias) as total "
        "FROM ocorrencias GROUP BY cidade "
        "ORDER BY total DESC LIMIT 10",
        conn
    )
print("Top 10 cidades via SQL:")
resultado

## 10. Interpretação dos Resultados

- **Distribuição regional:** A região Sudeste concentra o maior volume absoluto de ocorrências, reflexo da sua densidade populacional. Entretanto, regiões como Norte e Nordeste apresentam índices de violência médios mais elevados proporcionalmente.

- **Tipos de crime:** Roubo e furto lideram o ranking de frequência. Crimes como homicídio, embora menos frequentes, apresentam maior índice de violência associado.

- **Horários críticos:** O período noturno concentra crimes patrimoniais, enquanto violência doméstica se distribui de forma mais uniforme ao longo do dia.

- **Renda e violência:** A correlação entre renda média e índice de violência indica uma relação inversamente proporcional em certos estratos — regiões com menor renda tendem a apresentar indicadores mais elevados.

- **Tendência temporal:** A série histórica revela variações anuais associadas a ciclos econômicos e implementação de políticas de segurança pública.

## 11. Conclusão Executiva

Este projeto demonstrou que a análise de dados de criminalidade permite identificar padrões relevantes para o planejamento de políticas públicas de segurança. Os principais achados são:

1. A criminalidade não é uniforme — há concentração em cidades, regiões e períodos específicos.
2. Existe correlação negativa entre renda média e índice de violência.
3. O período noturno é criticamente mais perigoso para crimes patrimoniais.
4. A tendência histórica permite projeções e planejamento preventivo.

O dashboard interativo desenvolvido em Streamlit complementa esta análise com filtros dinâmicos e visualizações interativas, tornando os insights acessíveis a tomadores de decisão.